# Benchmarking Network Inference Methods on Transient Data

This notebook compares multiple methods on transient biological time-series data.


In [1]:
import numpy as np
import graph_tool.all as gt
import sys
import os
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, roc_curve, precision_recall_curve, auc
from scipy.stats import spearmanr



# Add the parent directory to sys.path so we can import our modules and gene_expr

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from inference_pipeline.data_generation.network_generator import generate_scale_free_W
from inference_pipeline.data_generation.simulator import simulate_realistic_pairs
from inference_pipeline.inference.mcmc_engine import run_burn_in, run_posterior_sampling, compute_ensemble_stats
from inference_pipeline.inference.metrics import plot_probabilistic_network, plot_inferred_vs_true_am, compute_reconstruction_metrics
from inference_pipeline.prediction.export_utils import export_ensemble_to_bin, export_true_to_bin
from gene_expr import GeneExprBlockState


In [2]:
N = 10
m = 1
seeds_network = [45, 100, 2023] # 3 different network topologies
dt_vals = [0.1, 0.2, 0.5, 1.0, 2.0] # 5 points for dt

dt_sim = 0.005
M_train = 50
M_test = 50
t_window = (0.0, 6.0)

mean_d = 0.5
mean_V = 2.0
D_base = 0.02

In [3]:
import pandas as pd
from tqdm.notebook import tqdm
import time
import os

def calc_all_metrics(y_true_binary, y_true_continuous, y_scores, y_inferred_continuous=None):
    mask = ~np.eye(N, dtype=bool)
    y_true_b = y_true_binary[mask]
    y_s = y_scores[mask]
    
    fpr, tpr, _ = roc_curve(y_true_b, y_s)
    roc_auc = auc(fpr, tpr)
    
    precision, recall, _ = precision_recall_curve(y_true_b, y_s)
    pr_auc = auc(recall, precision)
    
    spearman_corr, _ = spearmanr(np.abs(y_true_continuous[mask]), np.abs(y_s))
    
    sign_acc = np.nan
    if y_inferred_continuous is not None:
        true_edges_mask = (y_true_b == 1)
        if np.any(true_edges_mask):
            true_signs = np.sign(y_true_continuous[mask][true_edges_mask])
            inf_signs = np.sign(y_inferred_continuous[mask][true_edges_mask])
            sign_acc = np.mean(true_signs == inf_signs)
    
    return fpr, tpr, roc_auc, precision, recall, pr_auc, spearman_corr, sign_acc

results_list = []

for seed_network in tqdm(seeds_network, desc="Network Seeds"):
    # Generate TRUE network
    W_true = generate_scale_free_W(N, m, seed=seed_network)
    W_true_binary = (W_true != 0).astype(int)
    theta_true = [np.log(mean_d), np.log(mean_V), np.log(D_base)]
    
    for dt_meas in tqdm(dt_vals, desc=f"dt values (seed {seed_network})", leave=False):
        dt_sample = dt_meas
        
        # Simulate Training Data
        s_train, _, _, _, _ = simulate_realistic_pairs(
            W_true, M=M_train, dt_sim=dt_sim, dt_sample=dt_sample, dt_meas=dt_meas, 
            D_base=D_base, mean_d=mean_d, std_d=0.0, mean_V=mean_V, std_V=0.0, seed=100+seed_network, t_window=t_window
        )
        
        # --- Standard Likelihood ---
        g_init = gt.Graph(directed=True)
        g_init.add_vertex(N)
        num_initial_edges = max(1, int(N * N * 0.05))
        possible_edges = [(u, v) for u in range(N) for v in range(N) if u != v]
        np.random.seed(42)
        np.random.shuffle(possible_edges)
        for u, v in possible_edges[:num_initial_edges]:
            g_init.add_edge(u, v)

        initial_theta = [0.0, 0.0, np.log(0.05)]
        
        # State without EM
        state_std = GeneExprBlockState(s_train, dt=dt_meas, g=g_init.copy(), state_args=dict(deg_corr=False), theta=initial_theta.copy(), use_em_likelihood=False)
        state_std = run_burn_in(state_std, n_burnin=100, theta_mode="global")
        W_ensemble_std, _ = run_posterior_sampling(state_std, N, n_samples=100, delta_k=1, theta_mode="global")
        edge_prob_std, weight_mean_std, _ = compute_ensemble_stats(W_ensemble_std)
        scores_std = edge_prob_std * np.abs(weight_mean_std)
        _, _, roc_auc_std, _, _, pr_auc_std, spearman_std, _ = calc_all_metrics(W_true_binary, W_true, scores_std, weight_mean_std)
        
        results_list.append({
            'Seed': seed_network,
            'dt': dt_meas,
            'Method': 'Standard',
            'AUROC': roc_auc_std,
            'AUPRC': pr_auc_std,
            'Spearman': spearman_std
        })
        
        # --- EM Likelihood ---
        state_em = GeneExprBlockState(s_train, dt=dt_meas, g=g_init.copy(), state_args=dict(deg_corr=False), theta=initial_theta.copy(), use_em_likelihood=True)
        state_em = run_burn_in(state_em, n_burnin=100, theta_mode="global")
        W_ensemble_em, _ = run_posterior_sampling(state_em, N, n_samples=100, delta_k=1, theta_mode="global")
        edge_prob_em, weight_mean_em, _ = compute_ensemble_stats(W_ensemble_em)
        scores_em = edge_prob_em * np.abs(weight_mean_em)
        _, _, roc_auc_em, _, _, pr_auc_em, spearman_em, _ = calc_all_metrics(W_true_binary, W_true, scores_em, weight_mean_em)
        
        results_list.append({
            'Seed': seed_network,
            'dt': dt_meas,
            'Method': 'EM',
            'AUROC': roc_auc_em,
            'AUPRC': pr_auc_em,
            'Spearman': spearman_em
        })

# Save results
os.makedirs("data", exist_ok=True)
df_results = pd.DataFrame(results_list)
df_results.to_csv("data/likelihood_benchmark_results.csv", index=False)
display(df_results.head())

Network Seeds:   0%|          | 0/3 [00:00<?, ?it/s]

dt values (seed 45):   0%|          | 0/5 [00:00<?, ?it/s]


--- PHASE 1: Equilibration (100 sweeps) ---
Burn-in Sweep  1/100: Entropy = 235.8237 (dS_topo = -163.0033, dS_param = -20.0812, moves = 6564)
Burn-in Sweep 10/100: Entropy =  27.5358 (dS_topo =  18.0296, dS_param =  -1.9697, moves = 9148)
Burn-in Sweep 20/100: Entropy = -23.8535 (dS_topo =   3.3464, dS_param =  -1.4006, moves = 6616)
Burn-in Sweep 30/100: Entropy =  66.5569 (dS_topo =  19.2215, dS_param =  -0.6362, moves = 5754)
Burn-in Sweep 40/100: Entropy = 239.6246 (dS_topo = -24.4412, dS_param =  -0.4360, moves = 5464)
Burn-in Sweep 50/100: Entropy = 277.6844 (dS_topo = -58.5304, dS_param =  -0.1578, moves = 4963)
Burn-in Sweep 60/100: Entropy = 227.1751 (dS_topo =   5.9888, dS_param =  -0.2771, moves = 6183)
Burn-in Sweep 70/100: Entropy = 116.6835 (dS_topo = -36.8160, dS_param =  -0.1476, moves = 6788)
Burn-in Sweep 80/100: Entropy =  71.6232 (dS_topo = -40.6319, dS_param =  -0.0976, moves = 5592)
Burn-in Sweep 90/100: Entropy = -56.7710 (dS_topo =  -0.5297, dS_param =  -0.0854

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/likelihood_benchmark_results.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.lineplot(data=df, x='dt', y='AUROC', hue='Method', marker='o', ax=axes[0])
axes[0].set_title('AUROC vs dt')
axes[0].set_xlabel('dt (Sampling & Measurement)')
axes[0].set_ylabel('AUROC')

sns.lineplot(data=df, x='dt', y='AUPRC', hue='Method', marker='o', ax=axes[1])
axes[1].set_title('AUPRC vs dt')
axes[1].set_xlabel('dt (Sampling & Measurement)')
axes[1].set_ylabel('AUPRC')

plt.tight_layout()
plt.show()